In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import importlib.util

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# 0. Locate results folder
# ============================================================

try:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import get_project_paths
    RESULTS_DIR = get_project_paths(PROJECT_ROOT).results
except Exception:
    RESULTS_DIR = Path.cwd() / "results"

if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"No encuentro la carpeta results en: {RESULTS_DIR}")

FINAL_DIR = RESULTS_DIR / "final_summary_tables"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]


# ============================================================
# 1. Files to load
# ============================================================

COUNTRY_H_FILES = [
    ("baselines_ari_rolling_2023.csv", "baseline"),
    ("baselines_ili_rolling_2023.csv", "baseline"),

    ("sarima_ari_suite_rolling_2023.csv", "sarima"),
    ("sarima_ili_suite_rolling_2023.csv", "sarima"),

    ("random_forest_ari_rolling_2023.csv", "rf_local"),
    ("random_forest_ili_rolling_2023.csv", "rf_local"),

    ("rf_global_all_infections_all_countries_country_h.csv", "rf_pooled"),

    ("dl_candidates_country_h_2023.csv", "deep_learning"),

    ("bivariate_varima_country_h_2023.csv", "bivariate_varima"),
]

EXPECTED_FILES_PRIORITY = [
    "random_forest_ari_expected_points_2023.csv",
    "random_forest_ili_expected_points_2023.csv",
]


# ============================================================
# 2. Reading and normalization helpers
# ============================================================

def read_csv_safe(filename):
    path = RESULTS_DIR / filename
    if not path.exists():
        print(f"[WARN] No encontrado: {filename}")
        return None
    return pd.read_csv(path)


def normalize_country_h(df, source, filename):
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc == "mae":
            rename[c] = "MAE"
        elif lc == "rmse":
            rename[c] = "RMSE"
        elif lc == "mase":
            rename[c] = "MASE"
        elif lc == "n":
            rename[c] = "n"
        elif lc == "disease":
            rename[c] = "disease"
        elif lc == "location":
            rename[c] = "location"
        elif lc == "h":
            rename[c] = "h"
        elif lc == "model":
            rename[c] = "model"

    df = df.rename(columns=rename)

    required = ["disease", "location", "h", "model", "MAE", "RMSE", "MASE", "n"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{filename} no tiene columnas necesarias: {missing}. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    df = df[required].copy()

    df["source"] = source
    df["file"] = filename

    df["disease"] = df["disease"].astype(str)
    df["location"] = df["location"].astype(str)
    df["model"] = df["model"].astype(str)

    df["h"] = pd.to_numeric(df["h"], errors="coerce")
    df["MAE"] = pd.to_numeric(df["MAE"], errors="coerce")
    df["RMSE"] = pd.to_numeric(df["RMSE"], errors="coerce")
    df["MASE"] = pd.to_numeric(df["MASE"], errors="coerce")
    df["n"] = pd.to_numeric(df["n"], errors="coerce")

    df = df.dropna(
        subset=["disease", "location", "h", "model", "MAE", "RMSE", "MASE", "n"]
    ).copy()

    df["h"] = df["h"].astype(int)
    df["n"] = df["n"].astype(int)

    return df


# ============================================================
# 3. Load all country-horizon result tables
# ============================================================

parts = []
loaded_files = []

for filename, source in COUNTRY_H_FILES:
    df = read_csv_safe(filename)

    if df is None:
        continue

    norm = normalize_country_h(df, source, filename)
    parts.append(norm)
    loaded_files.append(filename)

if not parts:
    raise ValueError("No se ha cargado ninguna tabla country_h válida.")

all_country_h = pd.concat(parts, ignore_index=True)

print("Loaded country-h files:")
for f in loaded_files:
    print("-", f)

print("\nall_country_h shape:", all_country_h.shape)
display(all_country_h.head())


# ============================================================
# 4. Load expected points
# ============================================================

def load_expected_points():
    expected_parts = []

    for filename in EXPECTED_FILES_PRIORITY:
        df = read_csv_safe(filename)

        if df is None:
            continue

        df = df.copy()
        df.columns = [c.strip() for c in df.columns]

        rename = {}
        for c in df.columns:
            lc = c.lower()
            if lc == "disease":
                rename[c] = "disease"
            elif lc == "location":
                rename[c] = "location"
            elif lc == "h":
                rename[c] = "h"
            elif lc == "expected_actual_points":
                rename[c] = "expected_actual_points"

        df = df.rename(columns=rename)

        required = ["disease", "location", "h", "expected_actual_points"]
        missing = [c for c in required if c not in df.columns]

        if missing:
            raise ValueError(
                f"{filename} no tiene columnas necesarias: {missing}. "
                f"Columnas disponibles: {list(df.columns)}"
            )

        df = df[required].copy()
        expected_parts.append(df)

    if expected_parts:
        expected = pd.concat(expected_parts, ignore_index=True)
        expected = (
            expected
            .drop_duplicates(["disease", "location", "h"])
            .reset_index(drop=True)
        )

        expected["disease"] = expected["disease"].astype(str)
        expected["location"] = expected["location"].astype(str)
        expected["h"] = expected["h"].astype(int)
        expected["expected_actual_points"] = expected["expected_actual_points"].astype(int)

        return expected

    print(
        "[WARN] No encuentro expected_points de RF local. "
        "Derivo expected_points desde el mayor n disponible por disease/location/h."
    )

    tmp = (
        all_country_h
        .groupby(["disease", "location", "h"], as_index=False)
        .agg(expected_actual_points=("n", "max"))
    )

    return tmp


expected_points = load_expected_points()

print("\nExpected points by disease/h:")
display(
    expected_points
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        expected_countries=("location", "nunique")
    )
)


# ============================================================
# 5. Expected global and common6 coverage
# ============================================================

expected_global = (
    expected_points
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        expected_countries=("location", "nunique")
    )
)

expected_common6 = (
    expected_points[expected_points["location"].isin(COMMON6)]
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        expected_countries=("location", "nunique")
    )
)


# ============================================================
# 6. Build global summaries
# ============================================================

def build_horizon_summary(country_h):
    out = (
        country_h
        .groupby(["disease", "h", "source", "model"], as_index=False)
        .agg(
            MAE=("MAE", "mean"),
            RMSE=("RMSE", "mean"),
            MASE=("MASE", "mean"),
            n_countries=("location", "nunique"),
            n_points=("n", "sum")
        )
        .merge(expected_global, on=["disease", "h"], how="left")
    )

    out["coverage_ok"] = (
        (out["n_points"] == out["expected_actual_points"]) &
        (out["n_countries"] == out["expected_countries"])
    )

    out = out.sort_values(
        ["disease", "h", "coverage_ok", "MASE", "MAE"],
        ascending=[True, True, False, True, True]
    ).reset_index(drop=True)

    return out


global_horizon_all = build_horizon_summary(all_country_h)

global_horizon_full_coverage = (
    global_horizon_all[
        (global_horizon_all["coverage_ok"]) &
        (global_horizon_all["source"] != "bivariate_varima")
    ]
    .copy()
    .sort_values(["disease", "h", "MASE", "MAE"])
    .reset_index(drop=True)
)

global_top5 = (
    global_horizon_full_coverage
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

global_winners = (
    global_horizon_full_coverage
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(1)
    .reset_index(drop=True)
)


# ============================================================
# 7. Build common6 summaries
# ============================================================

common6_country_h = all_country_h[all_country_h["location"].isin(COMMON6)].copy()

common6_horizon_all = (
    common6_country_h
    .groupby(["disease", "h", "source", "model"], as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        MASE=("MASE", "mean"),
        n_countries=("location", "nunique"),
        n_points=("n", "sum")
    )
    .merge(expected_common6, on=["disease", "h"], how="left")
)

common6_horizon_all["coverage_ok"] = (
    (common6_horizon_all["n_points"] == common6_horizon_all["expected_actual_points"]) &
    (common6_horizon_all["n_countries"] == common6_horizon_all["expected_countries"])
)

common6_horizon_all = (
    common6_horizon_all
    .sort_values(
        ["disease", "h", "coverage_ok", "MASE", "MAE"],
        ascending=[True, True, False, True, True]
    )
    .reset_index(drop=True)
)

common6_horizon_full_coverage = (
    common6_horizon_all[common6_horizon_all["coverage_ok"]]
    .copy()
    .sort_values(["disease", "h", "MASE", "MAE"])
    .reset_index(drop=True)
)

common6_top5 = (
    common6_horizon_full_coverage
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

common6_winners = (
    common6_horizon_full_coverage
    .sort_values(["disease", "h", "MASE", "MAE"])
    .groupby(["disease", "h"], group_keys=False)
    .head(1)
    .reset_index(drop=True)
)


# ============================================================
# 8. Coverage audits
# ============================================================

coverage_audit_global = (
    all_country_h
    .groupby(["disease", "h", "source", "model"], as_index=False)
    .agg(
        n_countries=("location", "nunique"),
        n_points=("n", "sum")
    )
    .merge(expected_global, on=["disease", "h"], how="left")
)

coverage_audit_global["global_coverage_ok"] = (
    (coverage_audit_global["n_points"] == coverage_audit_global["expected_actual_points"]) &
    (coverage_audit_global["n_countries"] == coverage_audit_global["expected_countries"])
)

coverage_audit_common6 = (
    common6_country_h
    .groupby(["disease", "h", "source", "model"], as_index=False)
    .agg(
        n_countries=("location", "nunique"),
        n_points=("n", "sum")
    )
    .merge(expected_common6, on=["disease", "h"], how="left")
)

coverage_audit_common6["common6_coverage_ok"] = (
    (coverage_audit_common6["n_points"] == coverage_audit_common6["expected_actual_points"]) &
    (coverage_audit_common6["n_countries"] == coverage_audit_common6["expected_countries"])
)


# ============================================================
# 9. Final formatting and save
# ============================================================

def round_metrics(df):
    df = df.copy()
    for c in ["MAE", "RMSE", "MASE"]:
        if c in df.columns:
            df[c] = df[c].round(4)
    return df


tables = {
    "01_global_all_with_flags": round_metrics(global_horizon_all),
    "02_global_full_coverage": round_metrics(global_horizon_full_coverage),
    "03_global_top5": round_metrics(global_top5),
    "04_global_winners": round_metrics(global_winners),

    "05_common6_all_with_flags": round_metrics(common6_horizon_all),
    "06_common6_full_coverage": round_metrics(common6_horizon_full_coverage),
    "07_common6_top5": round_metrics(common6_top5),
    "08_common6_winners": round_metrics(common6_winners),

    "09_coverage_audit_global": coverage_audit_global,
    "10_coverage_audit_common6": coverage_audit_common6,

    "11_all_country_h_clean": all_country_h,
    "12_expected_points": expected_points,
}

for name, df in tables.items():
    out_path = FINAL_DIR / f"{name}.csv"
    df.to_csv(out_path, index=False)

print("\nCSV tables saved to:")
print(FINAL_DIR)


# ============================================================
# 11. Display key tables
# ============================================================

print("\n================ GLOBAL FULL-COVERAGE COMPARISON ================")
display(round_metrics(global_horizon_full_coverage))

print("\n================ GLOBAL WINNERS BY DISEASE/HORIZON ================")
display(round_metrics(global_winners))

print("\n================ COMMON6 FULL-COVERAGE COMPARISON ================")
display(round_metrics(common6_horizon_full_coverage))

print("\n================ COMMON6 WINNERS BY DISEASE/HORIZON ================")
display(round_metrics(common6_winners))

print("\n================ COVERAGE WARNINGS: GLOBAL ================")
bad_global = coverage_audit_global[~coverage_audit_global["global_coverage_ok"]].copy()
display(bad_global.sort_values(["source", "model", "disease", "h"]))

print("\n================ COVERAGE WARNINGS: COMMON6 ================")
bad_common6 = coverage_audit_common6[~coverage_audit_common6["common6_coverage_ok"]].copy()
display(bad_common6.sort_values(["source", "model", "disease", "h"]))

print("\nDone.")

Loaded country-h files:
- baselines_ari_rolling_2023.csv
- baselines_ili_rolling_2023.csv
- sarima_ari_suite_rolling_2023.csv
- sarima_ili_suite_rolling_2023.csv
- random_forest_ari_rolling_2023.csv
- random_forest_ili_rolling_2023.csv
- rf_global_all_infections_all_countries_country_h.csv
- dl_candidates_country_h_2023.csv
- bivariate_varima_country_h_2023.csv

all_country_h shape: (984, 10)


,disease,location,h,model,MAE,RMSE,MASE,n,source,file
0,ARI,BE,1,naive_last,148.748077,194.016179,0.501168,52,baseline,baselines_ari_rolling_2023.csv
1,ARI,BE,1,naive_year,296.859615,356.755570,1.000192,52,baseline,baselines_ari_rolling_2023.csv
2,ARI,BE,1,naive_mixed,148.748077,194.016179,0.501168,52,baseline,baselines_ari_rolling_2023.csv
3,ARI,BE,2,naive_last,173.172549,225.452898,0.583460,51,baseline,baselines_ari_rolling_2023.csv
4,ARI,BE,2,naive_year,292.923529,353.433305,0.986930,51,baseline,baselines_ari_rolling_2023.csv



Expected points by disease/h:


,disease,h,expected_actual_points,expected_countries
0,ARI,1,409,8
1,ARI,2,401,8
2,ARI,3,393,8
3,ARI,4,385,8
4,ILI,1,513,10
5,ILI,2,503,10
6,ILI,3,493,10
7,ILI,4,483,10



CSV tables saved to:
C:\Users\aolas\UNI PYTHON\TFG\results\final_summary_tables

[INFO] No se ha creado Excel porque no tienes instalado openpyxl ni xlsxwriter. No pasa nada: todas las tablas se han guardado como CSV.
Si quieres Excel, instala una de estas opciones:
pip install openpyxl
o
pip install xlsxwriter

================ GLOBAL FULL-COVERAGE COMPARISON ================


...


================ COVERAGE WARNINGS: GLOBAL ================


,disease,h,source,model,n_countries,n_points,expected_actual_points,expected_countries,global_coverage_ok
0,ARI,1,baseline,naive_last,8,402,409,8,False
14,ARI,2,baseline,naive_last,8,395,401,8,False
28,ARI,3,baseline,naive_last,8,387,393,8,False
42,ARI,4,baseline,naive_last,8,378,385,8,False
56,ILI,1,baseline,naive_last,10,506,513,10,False
70,ILI,2,baseline,naive_last,10,497,503,10,False
84,ILI,3,baseline,naive_last,10,487,493,10,False
98,ILI,4,baseline,naive_last,10,476,483,10,False
1,ARI,1,baseline,naive_mixed,8,402,409,8,False
15,ARI,2,baseline,naive_mixed,8,395,401,8,False



================ COVERAGE WARNINGS: COMMON6 ================


,disease,h,source,model,n_countries,n_points,expected_actual_points,expected_countries,common6_coverage_ok
0,ARI,1,baseline,naive_last,6,300,306,6,False
14,ARI,2,baseline,naive_last,6,295,300,6,False
28,ARI,3,baseline,naive_last,6,289,294,6,False
42,ARI,4,baseline,naive_last,6,282,288,6,False
56,ILI,1,baseline,naive_last,6,300,306,6,False
70,ILI,2,baseline,naive_last,6,295,300,6,False
84,ILI,3,baseline,naive_last,6,289,294,6,False
98,ILI,4,baseline,naive_last,6,282,288,6,False
1,ARI,1,baseline,naive_mixed,6,300,306,6,False
15,ARI,2,baseline,naive_mixed,6,295,300,6,False



Done.
